# Deliverable 3 — 02: Feature Engineering (24-Month Forecast)

**Objective:** Turn the American Airlines route-month dataset into model-ready features without revealing data that would not have been known at forecast time. **No model is trained here.**

**Forecast setup:** If the latest observed month is July 2026, a 24-month forecast predicts August 2026–July 2028, all at the **same July 2026 cutoff**. To support this, passenger-history features for a target month must come from **at least 24 calendar months earlier**. For example, a July 2028 forecast may use July 2026, but an August 2026 forecast's 24-month lag is August 2024.

This notebook produces **one supervised training row per observed route-month**. The final two-year forecast grid and model training belong in Deliverable 4. Because the available data is small, do not promise a high-accuracy network-wide forecast.

**Confidentiality:** The input/output datasets are NDA-protected. Save generated CSVs only under Git-ignored `data/`; clear notebook outputs before committing. Do not publish raw values, route-specific results, or charts without authorization.
**Additional experiment:** Historical route volatility, growth momentum, relative same-month demand, and history coverage using a strictly lagged 24-month window.


## 1. Load and validate local source data

Run using your project's `.venv` kernel. The notebook finds the repository's `.git` folder automatically, so it works from anywhere inside the cloned repository.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Open the notebook from inside the Git repository.")
    root = root.parent

source = root / "data" / "raw" / "AA_PAX_FORECASTING.csv"
if not source.exists():
    raise FileNotFoundError(f"Missing local NDA-protected CSV: {source}")

demand = pd.read_csv(source)
required = {"DEP_YEAR", "DEP_MONTH", "ORG_AIRPRT", "DES_AIRPRT", "OD_PAX"}
missing = required - set(demand.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print("Loaded route-month records:", len(demand))

## 2. Define target, route, and month

`OD_PAX` is the actual monthly passenger demand for **a specific directional origin–destination route**. We intentionally do **not** use `OD_REV_USD`, the exported index, or load timestamp as predictive features.

In [ ]:
data = demand[["DEP_YEAR", "DEP_MONTH", "ORG_AIRPRT", "DES_AIRPRT", "OD_PAX"]].copy()
data["DEP_YEAR"] = pd.to_numeric(data["DEP_YEAR"], errors="raise").astype(int)
data["DEP_MONTH"] = pd.to_numeric(data["DEP_MONTH"], errors="raise").astype(int)
data["OD_PAX"] = pd.to_numeric(data["OD_PAX"], errors="raise")
if not data["DEP_MONTH"].between(1, 12).all():
    raise ValueError("Unexpected month outside 1–12")
if data["OD_PAX"].isna().any() or (data["OD_PAX"] < 0).any():
    raise ValueError("Unexpected missing or negative passenger target")

data["ROUTE"] = data["ORG_AIRPRT"].astype(str) + " → " + data["DES_AIRPRT"].astype(str)
data["DATE"] = pd.to_datetime(dict(year=data["DEP_YEAR"], month=data["DEP_MONTH"], day=1))
if data.duplicated(["ROUTE", "DATE"]).any():
    raise ValueError("Duplicate route-month records: resolve before feature generation")
data = data.sort_values(["ROUTE", "DATE"]).reset_index(drop=True)
print("Unique routes:", data["ROUTE"].nunique())
print("Date coverage:", data["DATE"].min().date(), "to", data["DATE"].max().date())

## 3. Calendar and route features

The route and future month are known in advance. Cyclic encoding makes December and January close to one another. These are safe features for a 24-month forecast.

In [ ]:
features = data[["ROUTE", "DATE", "DEP_YEAR", "DEP_MONTH", "OD_PAX"]].copy()
features["MONTH_SIN"] = np.sin(2 * np.pi * features["DEP_MONTH"] / 12)
features["MONTH_COS"] = np.cos(2 * np.pi * features["DEP_MONTH"] / 12)
print("Calendar features created:", ["DEP_YEAR", "DEP_MONTH", "MONTH_SIN", "MONTH_COS", "ROUTE"])

## 4. Exact calendar-aligned passenger lags

A `.shift(24)` is **24 rows**, not necessarily 24 months. Missing months exist in the data. Instead, join passenger history by **route and exact calendar month**.

We generate 24-, 36-, and 48-month lags. At a fixed forecast cutoff, every value used is at least two years older than its target month, so these lag features are compatible with an **up-to-24-month forecast horizon**. Missing lags remain missing (we never fill missing demand with zero).

In [ ]:
history = data[["ROUTE", "DATE", "OD_PAX"]].copy()

for lag in [24, 36, 48]:
    lookup = history.copy()
    lookup["DATE"] = lookup["DATE"] + pd.DateOffset(months=lag)
    lookup = lookup.rename(columns={"OD_PAX": f"LAG_{lag}"})
    features = features.merge(lookup, on=["ROUTE", "DATE"], how="left", validate="one_to_one")

print("Calendar lag availability:")
print(features[["LAG_24", "LAG_36", "LAG_48"]].notna().sum())

## 5. Historical trend, rolling mean, and missingness flags

`RECENT_HIST_MEAN_3` averages months **24, 25, and 26 months before the target**, using only values known by the time a 24-month forecast would have been issued. Since months can be missing, it requires all three calendar months; otherwise it stays missing.

`HIST_CHANGE_12M` measures the change from 36 to 24 months prior. The model can learn whether a route was growing, declining, or missing sufficient history.

In [ ]:
# Create exact calendar offsets for 25 and 26 months as well.
for lag in [25, 26]:
    lookup = history.copy()
    lookup["DATE"] = lookup["DATE"] + pd.DateOffset(months=lag)
    lookup = lookup.rename(columns={"OD_PAX": f"LAG_{lag}"})
    features = features.merge(lookup, on=["ROUTE", "DATE"], how="left", validate="one_to_one")

features["RECENT_HIST_MEAN_3"] = features[["LAG_24", "LAG_25", "LAG_26"]].mean(axis=1, skipna=False)
features["HIST_CHANGE_12M"] = features["LAG_24"] - features["LAG_36"]
features["HIST_GROWTH_12M"] = (
    (features["LAG_24"] - features["LAG_36"]) /
    features["LAG_36"].replace(0, np.nan)
)
# Extreme ratios from very small routes can destabilize training; cap growth to +/- 300%.
features["HIST_GROWTH_12M"] = features["HIST_GROWTH_12M"].clip(-3, 3)
features["HAS_LAG_24"] = features["LAG_24"].notna().astype(int)

print("Rows with 24-month history:", int(features["HAS_LAG_24"].sum()))
print("Rows with all three rolling-history months:", int(features["RECENT_HIST_MEAN_3"].notna().sum()))

## 6. Market-aware route features (optional experiment)

We use a **24-month historical window ending 24 months before the target month** (exactly `LAG_24` through `LAG_47`). This guarantees these inputs were already known when forecasting up to 24 future months from one cutoff. We do **not** calculate statistics over the full dataset or use the target month's actual passengers.

New features:

- `ROUTE_VOLATILITY_24M`: standard deviation / mean of observed historic demand, when at least 12 of 24 months are available. Captures stability versus fluctuation.
- `ROUTE_MOMENTUM_12M`: difference between the average of lags 24–35 and lags 36–47, scaled by the older average. Captures historical growth or decline; requires at least 6 known months in each half.
- `ROUTE_SEASONALITY_INDEX`: historical demand in the same calendar month (lags 24 and 36), divided by mean demand across the historical window. Values above 1 indicate that month was stronger than average. Requires both corresponding months and at least 12 historic observations. This is a **noisy proxy**, not proven seasonality, because only two years contribute.
- `ROUTE_HISTORY_COVERAGE`: fraction of the last 24 historical months observed for this route. Distinguishes sparse from dense route history.

Missing historical observations are not silently filled with zero. These features are hypotheses to test on **earlier validation**, not confirmed accuracy improvements; the final test has already been inspected.

In [ ]:
# Every source value is from at least 24 calendar months before the target.
# Build exact-calendar lookups for offsets 24..47, reusing existing lag columns.
lag_columns = []
for lag in range(24, 48):
    name = f"LAG_{lag}"
    lag_columns.append(name)
    if name not in features.columns:
        lookup = history.rename(columns={"OD_PAX": name}).copy()
        lookup["DATE"] = lookup["DATE"] + pd.DateOffset(months=lag)
        features = features.merge(lookup, on=["ROUTE", "DATE"], how="left", validate="one_to_one")

window = features[lag_columns].astype(float)
window_n = window.notna().sum(axis=1)
window_mean = window.mean(axis=1)

# Coefficient of variation: values close to zero indicate steadier historical demand.
features["ROUTE_VOLATILITY_24M"] = (
    window.std(axis=1, ddof=0) / window_mean.replace(0, np.nan)
).where(window_n >= 12)

# Compare the two 12-month halves of the SAME available historical window.
recent = features[[f"LAG_{i}" for i in range(24, 36)]]
older = features[[f"LAG_{i}" for i in range(36, 48)]]
recent_avg = recent.mean(axis=1)
older_avg = older.mean(axis=1)
features["ROUTE_MOMENTUM_12M"] = (
    (recent_avg - older_avg) / older_avg.replace(0, np.nan)
).where((recent.notna().sum(axis=1) >= 6) & (older.notna().sum(axis=1) >= 6)).clip(-3, 3)

# Same-calendar-month demand compared with overall historic route demand.
features["ROUTE_SEASONALITY_INDEX"] = (
    features[["LAG_24", "LAG_36"]].mean(axis=1, skipna=False)
    / window_mean.replace(0, np.nan)
).where(window_n >= 12)

features["ROUTE_HISTORY_COVERAGE"] = window_n / 24.0

market_features = [
    "ROUTE_VOLATILITY_24M", "ROUTE_MOMENTUM_12M",
    "ROUTE_SEASONALITY_INDEX", "ROUTE_HISTORY_COVERAGE"
]
print("Engineered market-aware features:", market_features)
print("Nonmissing observations per new feature:")
print(features[market_features].notna().sum())

# Check bounds and prevent non-finite values entering the model-ready table.
assert features["ROUTE_HISTORY_COVERAGE"].between(0, 1).all()
for col in market_features:
    assert not np.isinf(features[col].to_numpy(dtype=float)).any(), f"Infinite values in {col}"

# Make sure the additional historical lag columns are not accidentally emitted as model inputs.
# They are only intermediate quantities used to calculate aggregate features.

## 7. Choose a reproducible model feature set

Route will be one-hot encoded during modeling, or passed to a preprocessing pipeline. We retain dates for the chronological split and `OD_PAX` as the label, **never as a same-month input**.

For the first comparable model, use only calendar and `LAG_24` inputs. Experiment with further history features after this simple version works. Keep incomplete historical features as `NaN`; tree models such as XGBoost can handle missing predictor values. A Random Forest implementation may need explicit imputation depending on the installed scikit-learn version.

In [ ]:
feature_columns = [
    "DEP_YEAR", "DEP_MONTH", "MONTH_SIN", "MONTH_COS", "ROUTE",
    "LAG_24", "LAG_36", "LAG_48", "LAG_25", "LAG_26",
    "RECENT_HIST_MEAN_3", "HIST_CHANGE_12M", "HIST_GROWTH_12M", "HAS_LAG_24",
    "ROUTE_VOLATILITY_24M", "ROUTE_MOMENTUM_12M",
    "ROUTE_SEASONALITY_INDEX", "ROUTE_HISTORY_COVERAGE"
]
model_table = features[["DATE", "ROUTE", "OD_PAX"] + [c for c in feature_columns if c != "ROUTE"]].copy()
assert len(model_table) == len(data)
assert not model_table.duplicated(["ROUTE", "DATE"]).any()

# This is the simple feature subset to try first for 24-month-ahead predictions.
initial_features = ["DEP_YEAR", "DEP_MONTH", "MONTH_SIN", "MONTH_COS", "ROUTE", "LAG_24"]
print("All engineered feature names:", feature_columns)
print("Recommended first experiment:", initial_features)
print("Number of model-ready rows:", len(model_table))

## 8. Automatic leakage and alignment checks

The crucial rule is that the historical demand attached to a target month is from **exactly 24 months earlier**, not 24 arbitrary rows earlier. We test a sample of present lags against original history. This also guards against accidental cross-route joins.

In [ ]:
check = features.loc[features["LAG_24"].notna(), ["ROUTE", "DATE", "LAG_24"]].copy()
check["SOURCE_DATE"] = check["DATE"] - pd.DateOffset(months=24)
original = history.rename(columns={"DATE": "SOURCE_DATE", "OD_PAX": "SOURCE_PAX"})
check = check.merge(original, on=["ROUTE", "SOURCE_DATE"], how="left", validate="one_to_one")
assert check["SOURCE_PAX"].notna().all(), "A historical reference is missing"
assert np.allclose(check["LAG_24"], check["SOURCE_PAX"]), "Calendar lag does not match historical data"
assert not {"OD_REV_USD", "SF_LOAD_TMS", "Unnamed: 0"}.intersection(feature_columns)
print("24-month historical alignment: PASS")
print("Excluded leakage-prone columns: PASS")
# Historical-window feature checks: all route statistics are built only from LAG_24..LAG_47.
assert all(int(c.split("_")[1]) >= 24 for c in lag_columns)
assert features["ROUTE_HISTORY_COVERAGE"].between(0, 1).all()
print("Market history window starts at 24 months before each target: PASS")

## 9. Save model-ready features locally (do not commit)

The CSV includes confidential passenger counts. It stays inside Git-ignored `data/processed/`. Verify `git status` does **not** show this dataset as a tracked or untracked file before pushing anything.

In [ ]:
output_path = root / "data" / "processed" / "aa_features_24m.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
model_table.to_csv(output_path, index=False)
print("Wrote local feature table under data/processed/ (do not commit)")
print("Rows:", len(model_table), "Columns:", len(model_table.columns))

## 10. Notes for Deliverable 4

- **Target:** `OD_PAX` for one route and one month.
- **Historical availability:** Route-specific observations have calendar gaps; `NaN` lags are missing history, not zero demand.
- **Valid for up to 24 months:** Features include calendar and route plus lags **24+ months old**. Each target month in the 24-month forecast can rely on those features as of a single cutoff immediately before the forecast begins.
- **Not yet a true forecast:** This notebook only builds supervised historical feature rows. Deliverable 4 must create the **future 24-month route × month grid**, join historical lags, train on past target months, evaluate with an appropriate historical cutoff, and compare against a horizon-compatible baseline.
- **Important validation nuance:** Standard one-year splits with `LAG_12` evaluate a different, shorter-horizon task. Report those separately; don't label their scores as 24-month forecast performance.
- **No real data in Git:** Clear notebook outputs before committing; keep all generated CSVs local and Git-ignored.
- **New features:** Market-aware statistics are historical proxies. Their actual benefit must be established with prior validation rather than the already-inspected holdout.
